Download All Auxiliary Data at **30m resolution** 
- CDL (Cropland Data Layer) — Arkansas & California
- Soil (POLARIS)
- Climate (Daymet)
- Topography (SRTM via Elevation API)


In [16]:
import requests, os, math, urllib3
import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.warp import calculate_default_transform, reproject, Resampling
from tqdm.notebook import tqdm


In [10]:

BASE_DIR = "C:/Users/chahi/Desktop/CNN_Transformer_Project/crop-classification/data/raw"
STATES = {
    'arkansas':   {'fips': '05', 'bbox': (-94.62, 33.00, -89.64, 36.50)},
    'california': {'fips': '06', 'bbox': (-124.41, 32.53, -114.13, 42.01)}
}

CDL_PATHS = {
    state: f'{BASE_DIR}/cdl/{state}/CDL_2021_{info["fips"]}.tif'
    for state, info in STATES.items()
}



## CDL — Cropland Data Layer


In [11]:

print(f"{'STATE':<12} | {'CRS':<10} | {'RES':<6} | {'SHAPE'}")
print("-" * 50)
for state, path in CDL_PATHS.items():
    if os.path.exists(path):
        try:
            with rasterio.open(path) as src:
                # src.res[0] pour avoir la valeur numérique de la résolution (ex: 30.0)
                res = src.res[0]
                print(f"{state.upper():<12} | {str(src.crs):<10} | {res:<5}m | {src.height}x{src.width}")
        except Exception as e:
            print(f"{state.upper():<12} | Erreur de lecture : {e}")
    else:
        print(f"{state.upper():<12} | ⚠️ Fichier introuvable à l'adresse : {path}")



STATE        | CRS        | RES    | SHAPE
--------------------------------------------------
ARKANSAS     | EPSG:5070  | 30.0 m | 13440x14847
CALIFORNIA   | EPSG:5070  | 30.0 m | 40345x23648


## Soil

In [ ]:
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ── variables POLARIS à télécharger ──────────────────────────────────────
SOIL_VARS   = ['clay', 'sand', 'silt', 'ph', 'om', 'bd'] 
SOIL_DEPTH  = '0_5'
SOIL_STAT   = 'mean'

# ⚠️ HTTP (pas HTTPS) — le serveur Duke ne supporte pas HTTPS pour ces fichiers
POLARIS_BASE = 'http://hydrology.cee.duke.edu/POLARIS/PROPERTIES/v1.0'

def polaris_url(var, depth, lat, lon, stat='mean'):
    """Format réel : .../v1.0/{var}/{stat}/{depth}/lat{S}{N}_lon{W}{E}.tif
       - pas de zero-padding sur longitude
       - HTTP obligatoire
    """
    lat_str = f'lat{abs(lat)}{abs(lat+1)}'
    lon_str = f'lon{lon}{lon+1}'   # ex: lon-95-94 (pas -095-094)
    return f'{POLARIS_BASE}/{var}/{stat}/{depth}/{lat_str}_{lon_str}.tif'

def download_file(url, out_path, desc=''):
    if os.path.exists(out_path):
        return True
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    try:
        r = requests.get(url, stream=True, timeout=120, headers=headers)
        
        if r.status_code == 404:
            print(f"      [404] {url}")
            return False
        r.raise_for_status()
        
        # ← tqdm sur les BYTES, pas sur les tuiles
        total = int(r.headers.get('content-length', 0))
        with open(out_path, 'wb') as f, tqdm(
            total=total, unit='B', unit_scale=True, unit_divisor=1024,
            desc=f'    {desc}', leave=False
        ) as bar:
            for chunk in r.iter_content(chunk_size=65536):
                f.write(chunk)
                bar.update(len(chunk))
        return True
        
    except Exception as e:
        print(f'    ⚠️ Échec {url}: {e}')
        if os.path.exists(out_path):
            os.remove(out_path)
        return False
    
def reproject_to_5070(src_path, dst_path, resolution=30):
    """Reprojette un GeoTIFF vers EPSG:5070 à 30m."""
    with rasterio.open(src_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, 'EPSG:5070', src.width, src.height,
            *src.bounds, resolution=resolution
        )
        kwargs = src.meta.copy()
        kwargs.update({'crs': 'EPSG:5070', 'transform': transform,
                       'width': width, 'height': height})
        with rasterio.open(dst_path, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs='EPSG:5070',
                    resampling=Resampling.bilinear
                )

def get_polaris_tiles(bbox):
    west, south, east, north = bbox
    lats = range(math.floor(south), math.ceil(north))
    lons = range(math.floor(west),  math.ceil(east))
    return [(lat, lon) for lat in lats for lon in lons]


# ── téléchargement POLARIS ────────────────────────────────────────────────
SOIL_PATHS = {}

for state, info in STATES.items():
    print(f'\n📥 Soil — {state.upper()}')
    tiles  = get_polaris_tiles(info['bbox'])
    state_dir = f'{BASE_DIR}/soil/{state}'
    os.makedirs(state_dir, exist_ok=True)
    SOIL_PATHS[state] = {}

    for var in SOIL_VARS:
        print(f'  Variable : {var}')
        tile_paths = []

        for lat, lon in tqdm(tiles, desc=f'  {var} tiles'):
            url      = polaris_url(var, SOIL_DEPTH, lat, lon)
            tmp_path = f'{state_dir}/{var}_{lat}_{lon}_wgs84.tif'
            ok = download_file(url, tmp_path, desc=f'{var}_{lat}_{lon}')
            if ok:
                tile_paths.append(tmp_path)

        if not tile_paths:
            print(f'    ⚠️ Aucune tuile trouvée pour {var}')
            continue

        # merge des tuiles
        merged_path = f'{state_dir}/{var}_merged_wgs84.tif'
        if not os.path.exists(merged_path):
            datasets = [rasterio.open(p) for p in tile_paths]
            mosaic, out_trans = merge(datasets)
            out_meta = datasets[0].meta.copy()
            out_meta.update({'height': mosaic.shape[1], 'width': mosaic.shape[2],
                             'transform': out_trans})
            with rasterio.open(merged_path, 'w', **out_meta) as dest:
                dest.write(mosaic)
            for ds in datasets:
                ds.close()

        # reprojection → EPSG:5070 @ 30m
        final_path = f'{state_dir}/{var}_5070_30m.tif'
        if not os.path.exists(final_path):
            reproject_to_5070(merged_path, final_path, resolution=30)

        SOIL_PATHS[state][var] = final_path
        print(f'    ✓ {final_path}')

print('\n✅ Soil terminé !')


📥 Soil — ARKANSAS
  Variable : clay


  clay tiles:   0%|          | 0/24 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Climate 

In [ ]:
import requests, os
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from tqdm.notebook import tqdm

# ── config Daymet ─────────────────────────────────────────────────────────
CLIMATE_YEAR = 2021
# variables dispo : tmax, tmin, prcp, srad, vp, swe, dayl
CLIMATE_VARS = ['tmax', 'tmin', 'prcp', 'srad', 'vp']

# API Daymet single-pixel ou tuile régionale
# On utilise l'API NetCDF régionale de ORNL DAAC
DAYMET_URL = 'https://thredds.daac.ornl.gov/thredds/ncss/ornldaac/2129'

def download_daymet(var, bbox, year, out_dir):
    """
    Télécharge Daymet annual summary (GeoTIFF) via ORNL THREDDS.
    Utilise le service NetCDF Subset Service.
    """
    west, south, east, north = bbox
    out_path = os.path.join(out_dir, f'daymet_{var}_{year}.nc')

    if os.path.exists(out_path):
        print(f'    {var} : déjà téléchargé ✓')
        return out_path

    # URL Daymet annual mosaic
    url = (
        f'https://thredds.daac.ornl.gov/thredds/ncss/ornldaac/2129/'
        f'daymet_v4_ann_{var}_na_{year}.nc'
        f'?var={var}'
        f'&north={north}&south={south}&east={east}&west={west}'
        f'&disableLLSubset=on&disableProjSubset=on'
        f'&horizStride=1&accept=netcdf'
    )

    try:
        r = requests.get(url, stream=True, timeout=300)
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(out_path, 'wb') as f, tqdm(
            total=total, unit='B', unit_scale=True, desc=f'    {var}'
        ) as bar:
            for chunk in r.iter_content(chunk_size=65536):
                f.write(chunk)
                bar.update(len(chunk))
        print(f'    ✓ {out_path}')
        return out_path
    except Exception as e:
        print(f'    ⚠️ Échec {var}: {e}')
        return None

def nc_to_tif_5070(nc_path, var, out_path, resolution=30):
    """Convertit NetCDF → GeoTIFF reprojeté en EPSG:5070 à 30m."""
    import subprocess
    # gdal_translate + gdalwarp via subprocess
    tmp = nc_path.replace('.nc', '_tmp.tif')
    subprocess.run([
        'gdal_translate', '-of', 'GTiff',
        f'NETCDF:{nc_path}:{var}', tmp
    ], check=True, capture_output=True)

    subprocess.run([
        'gdalwarp', '-t_srs', 'EPSG:5070',
        '-tr', str(resolution), str(resolution),
        '-r', 'bilinear',
        '-overwrite', tmp, out_path
    ], check=True, capture_output=True)

    os.remove(tmp)
    return out_path


# ── téléchargement Climate ────────────────────────────────────────────────
CLIMATE_PATHS = {}

for state, info in STATES.items():
    print(f'\n📥 Climate — {state.upper()}')
    state_dir = f'{BASE_DIR}/climate/{state}'
    os.makedirs(state_dir, exist_ok=True)
    CLIMATE_PATHS[state] = {}

    for var in CLIMATE_VARS:
        nc_path = download_daymet(var, info['bbox'], CLIMATE_YEAR, state_dir)
        if nc_path:
            tif_path = f'{state_dir}/{var}_5070_30m.tif'
            if not os.path.exists(tif_path):
                print(f'    Reprojection {var} → EPSG:5070 @ 30m...')
                try:
                    nc_to_tif_5070(nc_path, var, tif_path, resolution=30)
                    print(f'    ✓ {tif_path}')
                except Exception as e:
                    print(f'    ⚠️ Reprojection échouée: {e}')
                    continue
            CLIMATE_PATHS[state][var] = tif_path

print('\n✅ Climate terminé !')

## Topography

In [ ]:
import requests, os, subprocess
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from tqdm.notebook import tqdm

# ── OpenTopography API — SRTM GL1 (30m) ──────────────────────────────────
# API key gratuite sur https://opentopography.org/
OT_API_KEY = '8ddac409d51d0bdcfa251cc0237d98cb'  
OT_URL     = 'https://portal.opentopography.org/API/globaldem'

def download_srtm(bbox, out_dir, state_name, api_key=''):
    """
    Télécharge le DEM SRTM GL1 (30m) via OpenTopography API.
    """
    west, south, east, north = bbox
    out_path = os.path.join(out_dir, f'srtm_{state_name}_wgs84.tif')

    if os.path.exists(out_path):
        print(f'  {state_name} : déjà téléchargé ✓')
        return out_path

    params = {
        'demtype':    'SRTMGL1',   # 30m
        'south':      south,
        'north':      north,
        'west':       west,
        'east':       east,
        'outputFormat': 'GTiff',
    }
    if api_key:
        params['API_Key'] = api_key

    print(f'  Téléchargement SRTM {state_name}...')
    r = requests.get(OT_URL, params=params, stream=True, timeout=600)
    r.raise_for_status()

    total = int(r.headers.get('content-length', 0))
    with open(out_path, 'wb') as f, tqdm(
        total=total, unit='B', unit_scale=True, desc=f'  SRTM {state_name}'
    ) as bar:
        for chunk in r.iter_content(chunk_size=65536):
            f.write(chunk)
            bar.update(len(chunk))

    print(f'  ✓ {out_path} ({os.path.getsize(out_path)/1e6:.1f} MB)')
    return out_path

def compute_terrain_derivatives(dem_path, out_dir, state_name, resolution=30):
    """
    Calcule : élévation, pente (slope), aspect — reprojetés en EPSG:5070 @ 30m.
    """
    results = {}

    # reprojection DEM → EPSG:5070 @ 30m
    dem_5070 = os.path.join(out_dir, f'dem_{state_name}_5070_30m.tif')
    if not os.path.exists(dem_5070):
        print(f'  Reprojection DEM → EPSG:5070 @ 30m...')
        subprocess.run([
            'gdalwarp', '-t_srs', 'EPSG:5070',
            '-tr', str(resolution), str(resolution),
            '-r', 'bilinear', '-overwrite',
            dem_path, dem_5070
        ], check=True, capture_output=True)
    results['elevation'] = dem_5070
    print(f'  ✓ Elevation : {dem_5070}')

    # calcul de la pente (slope)
    slope_path = os.path.join(out_dir, f'slope_{state_name}_5070_30m.tif')
    if not os.path.exists(slope_path):
        print(f'  Calcul slope...')
        subprocess.run([
            'gdaldem', 'slope', dem_5070, slope_path,
            '-of', 'GTiff'
        ], check=True, capture_output=True)
    results['slope'] = slope_path
    print(f'  ✓ Slope : {slope_path}')

    # calcul de l'aspect
    aspect_path = os.path.join(out_dir, f'aspect_{state_name}_5070_30m.tif')
    if not os.path.exists(aspect_path):
        print(f'  Calcul aspect...')
        subprocess.run([
            'gdaldem', 'aspect', dem_5070, aspect_path,
            '-of', 'GTiff'
        ], check=True, capture_output=True)
    results['aspect'] = aspect_path
    print(f'  ✓ Aspect : {aspect_path}')

    return results


# ── téléchargement Topography ─────────────────────────────────────────────
TOPO_PATHS = {}

for state, info in STATES.items():
    print(f'\n📥 Topography — {state.upper()}')
    state_dir = f'{BASE_DIR}/topography/{state}'
    os.makedirs(state_dir, exist_ok=True)

    dem_path = download_srtm(info['bbox'], state_dir, state, api_key=OT_API_KEY)
    if dem_path:
        TOPO_PATHS[state] = compute_terrain_derivatives(dem_path, state_dir, state)

print('\n✅ Topography terminé !')

## vérification final

In [ ]:
import rasterio, os

def check_file(path, name):
    if path and os.path.exists(path):
        with rasterio.open(path) as src:
            print(f'  ✅ {name}')
            print(f'     CRS : {src.crs}  |  Res : {src.res}m  |  Shape : {src.height}x{src.width}')
    else:
        print(f'  ❌ {name} — MANQUANT')

print('=' * 60)
print('RÉSUMÉ DES DONNÉES TÉLÉCHARGÉES')
print('=' * 60)

for state in STATES:
    print(f'\n── {state.upper()} ──')

    print('  CDL:')
    check_file(CDL_PATHS.get(state), 'CDL 2021')

    print('  Soil:')
    for var, path in SOIL_PATHS.get(state, {}).items():
        check_file(path, f'Soil {var}')

    print('  Climate:')
    for var, path in CLIMATE_PATHS.get(state, {}).items():
        check_file(path, f'Climate {var}')

    print('  Topography:')
    for var, path in TOPO_PATHS.get(state, {}).items():
        check_file(path, f'Topo {var}')

print('\n' + '='*60)
print('Tous les fichiers sont en EPSG:5070 @ 30m ✓')